In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model = "gpt-4o-mini")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 잘 지냈어요! 당신은요? 어떤 도움을 드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 12, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BkQeQgV0lpWb3DpVshQDUk83NWN33', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--14a7ed64-4e23-49be-a42b-2584f6d2b3f0-0', usage_metadata={'input_tokens': 12, 'output_tokens': 18, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool
def get_current_time(timezone:str, location:str) -> str:
    """현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존(예: 'Asia/Seoul'). 실제 존재해야 함.
        location (str): 지역명. 타임존은 모든 지명에 대응되지 않으므로 이후 llm 답변 생성에 사용됨

    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재 시각 {now}'

    print(location_and_local_time)
    return location_and_local_time

In [4]:
tools = [get_current_time]

tool_dict = {"get_current_time" : get_current_time}

llm_with_tools = llm.bind_tools(tools)

In [5]:
from langchain_core.messages import SystemMessage

messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇 시야?")
]

response = llm_with_tools.invoke(messages)
messages.append(response)

print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_jd0Zcbq6eOIW2ZDEDYmIl5e0', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"부산"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 129, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BkQopgRz3ndet6JStBlpz3Yasb6WA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--b8f7c048-b9b3-4180-9ee2-785e6db42115-0', tool_call

In [6]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call['name']]
    print(tool_call['args'])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재 시각 2025-06-20 16:53:06


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_jd0Zcbq6eOIW2ZDEDYmIl5e0', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"부산"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 129, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BkQopgRz3ndet6JStBlpz3Yasb6WA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--b8f7c048-b9b3-4180-9ee2-785e6db42115-0', tool_ca

In [7]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 2025년 6월 20일 16시 53분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 185, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BkQrT3NSzkQq3ESvv0BNcCoU4Znfg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--c1f3fb0d-b8ef-4790-b109-ede0a38102f7-0', usage_metadata={'input_tokens': 185, 'output_tokens': 23, 'total_tokens': 208, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [8]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker:str = Field(..., title="주식 코드", description = "주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description = "주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [9]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown()

    return history_md


tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time" : get_current_time, "get_yf_stock_history" : get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)


    

In [10]:
messages.append(HumanMessage("테슬라는 한 달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'tool_calls': [{'id': 'call_2C6FmMPL1zYNQpot9PQPT6AL', 'function': {'arguments': '{"stock_history_input":{"ticker":"TSLA","period":"1mo"}}', 'name': 'get_yf_stock_history'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 276, 'total_tokens': 303, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BkQxjzDOz58yV9YBq0KeCDdkYaJKB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--e441c0f8-6efa-48db-ac2e-4f3b60b0c04f-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_2C6FmMPL1zYNQpot9PQPT6AL', 'type': 'tool_call'}] usage_metadata={'inp

In [11]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)
    

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2025-05-19 00:00:00-04:00 | 336.3  | 343    | 333.37 |  342.09 | 8.88699e+07 |           0 |              0 |\n| 2025-05-20 00:00:00-04:00 | 347.87 | 354.99 | 341.63 |  343.82 | 1.31716e+08 |           0 |              0 |\n| 2025-05-21 00:00:00-04:00 | 344.43 | 347.35 | 332.2  |  334.62 | 1.02355e+08 |           0 |              0 |\n| 2025-05-22 00:00:00-04:00 | 331.9  | 347.27 | 331.39 |  341.04 | 9.71134e+07 |           0 |              0 |\n| 2025-05-23 00:00:00-04:00 | 337.92 | 343.18 | 333.21 |  339.34 | 8.46548e+07 |           0 |              0 |\n| 2025-05-27 00:00:00-04:00 | 347.35 | 363.79 | 347.32 |  362.89 | 1.20146e+08 |           0 |              0 |\n| 2025-05-28 00:00:00-04:0

In [12]:
llm_with_tools.invoke(messages)

AIMessage(content='한 달 전인 2025년 5월 19일 테슬라의 주가는 342.09 달러로 마감했습니다. 현재 날짜인 2025년 6월 20일의 주가는 확인할 수 없지만, 최근의 데이터를 통해 주가가 지난 달과 비교하여 하락세를 보이고 있음을 알 수 있습니다. 예를 들어, 6월 18일의 주가는 322.05 달러입니다.\n\n결론적으로, 테슬라의 주가는 한 달 전에 비해 하락했습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 1635, 'total_tokens': 1753, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BkQzdujI9ZMGHHopFs3kFpSAllSaU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--564b7de6-f753-4301-bb8e-f4ee1ba290cf-0', usage_metadata={'input_tokens': 1635, 'output_tokens': 118, 'total_tokens': 1753, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reas